# logit-thermo A2-3：分析（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 2〜5 分。
- `MyDrive/logit-thermo/A2_3/` の 6 つの条件と、A2-2 の `qwen2.5-1.5b.npz`（A100 の確かめ用）を読み、ゲート v0.2 の判定（本命・参考・軸ごとの読み方）を計算して `A2_3/a2_3_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：`scripts/a2_2_analyze.py`（`35142703f6b88d7a…`）と `scripts/a2_3_analyze.py`（`6003f2fc37ea03ae…`）をそのまま埋め込み（データを開く前に固定）。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-3 分析（判定と要約、CPU）」。終わったらランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-3 分析（判定と要約、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-3 analyze', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：分析のコード
import hashlib
SRCA22 = '"""A2-2（ゲート v0.3、凍結 2026-10-27 21:34）の分析。データを開く前に書いて固定する。\n\n    python3 scripts/a2_2_analyze.py <A2_2 のフォルダ> <出力 json>\n\nゲートに書いていない細部（データを開く前にここで決めた。結果ノートにも写す）：\n- F(T)：書き出しごとに、生成 2 本の ℓ_gen を平均してから ℓ_tf を引き、t = 2〜512 で平均。書き出しどうしの中央値。\n- ε：格子の各温度で、書き出し単位のブートストラップ（2,000 回、種 0）による F の 95% 区間の半分の幅。その中央値。\n  T_0 のブートストラップでは ε を全データの値に固定する（入れ子のブートストラップはしない）。\n  交差しない回は区間の計算から外し、その割合を記録する。\n- T_cal：全書き出しの Σ(⟨z⟩_T − z_y) を足し、符号が + から − に変わるところを対数温度で直線補間。\n  有限でない値を含む書き出し（人のトークンが外した番号だった場合）は T_cal から外して記録する。\n- a：a_loc の、全位置・全書き出しの中央値。\n- 対照 B は「ほかの家族の T_0 の平均（算術）」、C は「a × ほかの家族の T_0 ÷ a の平均」。どちらも交差のあるモデルだけで作る。\n- E2 の「誤差 10% 以内」は |log(T_0 ÷ T_cal)| ≤ log(1.10)。「区間の幅 15% 以下」は 区間の上 ÷ 下 ≤ 1.15。\n- E1：どちらかの組で T_0 がない（交差なし）ときは「交差なしで判定できない」と書く。\n- P2：横軸 x1 = log T、x3 = ℓ_tf(T)（全位置・全書き出しの中央値）。全 7 モデルの 15 点を使う。\n  3 次の最小二乗スプライン（scipy LSQUnivariateSpline、k = 3）。内側の結び目は 5 つで、学習に使う x の分位点\n  （1/6, …, 5/6）に置く。x1 と x3 で柔らかさをそろえるため。同じ x は平均してまとめる。\n  学習の範囲の外は端の値で止める（ext = \'const\'）。範囲の外の点の割合を記録する。\n  （最初は平滑化スプライン s = 点の数 × σ² にしていたが、作り物のデータの確かめで、x3 の近い点が並ぶと\n   スプラインが暴れて x3 だけが不公平に負けることが分かったので、データを開く前に替えた。）\n  誤差は「抜いた家族の各モデルの二乗誤差の平均」の、全モデルでの平均。\n- Q6（記録）：自分の文への特別な反応 A = |ℓ_gen − ℓ_cross|、変な文だから B = |ℓ_cross − ℓ_tf(文字の窓)|。\n  ℓ_gen は読み手自身の生成の t = 2〜512 の平均、ℓ_cross と ℓ_tf(文字の窓) は対照のファイルの値。\n  4 つの向き × 15 温度で平均した A と B を比べ、|A − B| ≤ 0.2 × max(A, B) なら「同じくらい」。\n"""\nimport json, math, os, sys\nimport numpy as np\nfrom scipy.interpolate import LSQUnivariateSpline\n\nMODELS = [(\'pythia-1b\', \'Pythia\', \'base\'), (\'qwen2.5-0.5b\', \'Qwen\', \'base\'), (\'qwen2.5-0.5b-inst\', \'Qwen\', \'inst\'),\n          (\'qwen2.5-1.5b\', \'Qwen\', \'base\'), (\'olmo2-1b\', \'OLMo\', \'base\'), (\'smollm2-1.7b\', \'SmolLM\', \'base\'),\n          (\'smollm2-1.7b-inst\', \'SmolLM\', \'inst\')]\nPAIRS = [(\'qwen2.5-0.5b\', \'qwen2.5-0.5b-inst\'), (\'smollm2-1.7b\', \'smollm2-1.7b-inst\')]\nCROSS = [(\'pythia-1b\', \'olmo2-1b\'), (\'olmo2-1b\', \'pythia-1b\'), (\'qwen2.5-1.5b\', \'smollm2-1.7b\'), (\'smollm2-1.7b\', \'qwen2.5-1.5b\')]\nNBOOT, SEED = 2000, 0\nLOG110 = math.log(1.10)\n\n\ndef pav(y):\n    blocks = [[float(v), 1] for v in y]\n    i = 0\n    while i < len(blocks) - 1:\n        if blocks[i][0] > blocks[i + 1][0]:\n            m = (blocks[i][0] * blocks[i][1] + blocks[i + 1][0] * blocks[i + 1][1]) / (blocks[i][1] + blocks[i + 1][1])\n            blocks[i] = [m, blocks[i][1] + blocks[i + 1][1]]; del blocks[i + 1]\n            i = max(i - 1, 0)\n        else:\n            i += 1\n    out = []\n    for m, n in blocks:\n        out += [m] * n\n    return np.array(out)\n\n\ndef crossing_mono(F, temps, eps):\n    g = pav(np.asarray(F, float))\n    neg = np.nonzero(g <= -eps)[0]; pos = np.nonzero(g >= eps)[0]\n    if len(neg) == 0 or len(pos) == 0 or pos[0] < neg[-1]:\n        return float(\'nan\')\n    i, j = neg[-1], pos[0]\n    lt0, lt1 = math.log(temps[i]), math.log(temps[j])\n    return float(math.exp(lt0 + (0 - g[i]) / (g[j] - g[i]) * (lt1 - lt0)))\n\n\ndef per_doc_feedback(tf_q, gen_q):\n    """tf_q：(nD, nT, L)、gen_q：(nT, S, nD, L) → (nD, nT)：t = 2〜L の f の平均。"""\n    g = gen_q.astype(np.float64).mean(1).transpose(1, 0, 2)        # (nD, nT, L)\n    f = g - tf_q.astype(np.float64)\n    return f[:, :, 1:].mean(-1)\n\n\ndef feedback_stats(fd, temps, rng):\n    nD = fd.shape[0]\n    F = np.median(fd, 0)\n    idx = rng.integers(0, nD, size=(NBOOT, nD))\n    Fb = np.median(fd[idx], 1)                                       # (NBOOT, nT)\n    half = (np.percentile(Fb, 97.5, 0) - np.percentile(Fb, 2.5, 0)) / 2\n    eps = float(np.median(half))\n    T0 = crossing_mono(F, temps, eps)\n    T0b = np.array([crossing_mono(Fb[b], temps, eps) for b in range(NBOOT)])\n    ok = np.isfinite(T0b)\n    ci = [float(np.percentile(T0b[ok], 2.5)), float(np.percentile(T0b[ok], 97.5))] if ok.sum() >= 20 else [float(\'nan\')] * 2\n    return dict(F=F.tolist(), F_ci_half=half.tolist(), eps=eps, F_pav=pav(F).tolist(), T0=T0, T0_ci=ci,\n                boot_cross_frac=float(ok.mean()))\n\n\ndef t_cal(tcal, tcal_T):\n    fin = np.isfinite(tcal).all(1)\n    S = tcal[fin].sum(0)\n    lt = np.log(tcal_T)\n    i = np.nonzero((S[:-1] > 0) & (S[1:] <= 0))[0]\n    if len(i) == 0:\n        return float(\'nan\'), int((~fin).sum())\n    i = i[0]\n    return float(math.exp(lt[i] + S[i] / (S[i] - S[i + 1]) * (lt[i + 1] - lt[i]))), int((~fin).sum())\n\n\ndef distinct4(toks):\n    """toks：(nT, S, nD, L) → (nT,) 後半 128 トークンの distinct-4 の平均。"""\n    nT, S, nD, L = toks.shape\n    out = np.zeros(nT)\n    for k in range(nT):\n        v = []\n        for s in range(S):\n            for d in range(nD):\n                x = toks[k, s, d, L - 128:]\n                grams = {tuple(x[i:i + 4]) for i in range(len(x) - 3)}\n                v.append(len(grams) / (len(x) - 3))\n        out[k] = np.mean(v)\n    return out.tolist()\n\n\ndef spline_error(xs, Fs, fams, sigma):\n    errs, outside = [], []\n    for f in sorted(set(fams)):\n        tr = [i for i, g in enumerate(fams) if g != f]\n        te = [i for i, g in enumerate(fams) if g == f]\n        x = np.concatenate([xs[i] for i in tr]); y = np.concatenate([Fs[i] for i in tr])\n        ux = np.unique(x)\n        uy = np.array([y[x == v].mean() for v in ux])\n        knots = np.quantile(ux, np.arange(1, 6) / 6)\n        sp = LSQUnivariateSpline(ux, uy, knots, k=3, ext=\'const\')\n        for i in te:\n            errs.append(float(np.mean((sp(xs[i]) - Fs[i]) ** 2)))\n            outside.append(float(np.mean((xs[i] < ux.min()) | (xs[i] > ux.max()))))\n    return float(np.mean(errs)), float(np.mean(outside))\n\n\ndef judge(models, temps, q_tag):\n    """models：name → dict(fam, kind, fb, Tcal, a, ell_tf)。E1・E2・P2 と判定不能の条件。"""\n    out = {}\n    have = {n: m for n, m in models.items() if np.isfinite(m[\'fb\'][\'T0\'])}\n    T0s = {n: m[\'fb\'][\'T0\'] for n, m in have.items()}\n    # 判定不能の条件\n    if len(T0s) >= 2:\n        rng_ratio = max(T0s.values()) / min(T0s.values())\n        ci_ratios = [m[\'fb\'][\'T0_ci\'][1] / m[\'fb\'][\'T0_ci\'][0] for m in have.values() if np.isfinite(m[\'fb\'][\'T0_ci\'][0])]\n        typ = float(np.median(ci_ratios)) if ci_ratios else float(\'nan\')\n        undec_range = bool(np.isfinite(typ) and rng_ratio < typ)\n    else:\n        rng_ratio, typ, undec_range = float(\'nan\'), float(\'nan\'), True\n    tc = [m[\'Tcal\'] for m in models.values() if np.isfinite(m[\'Tcal\'])]\n    undec_D = bool(max(tc) / min(tc) < 1.10)\n    out[\'undecidable\'] = dict(T0_range_ratio=rng_ratio, T0_ci_ratio_typical=typ, E2_P2_undecidable=undec_range,\n                              Tcal_range_ratio=max(tc) / min(tc), E_vs_D_undecidable=undec_D)\n    # E1\n    e1 = []\n    for b, i in PAIRS:\n        dcal = math.log(models[i][\'Tcal\'] / models[b][\'Tcal\'])\n        if b in have and i in have:\n            d0 = math.log(T0s[i] / T0s[b])\n            ok = bool(np.sign(d0) == np.sign(dcal) and abs(d0 - dcal) <= 0.5 * abs(dcal))\n        else:\n            d0, ok = float(\'nan\'), None\n        e1.append(dict(pair=[b, i], d_cal=dcal, d_0=d0, ok=ok))\n    if any(p[\'ok\'] is None for p in e1):\n        v = \'交差なしで判定できない\'\n    elif all(abs(p[\'d_cal\']) < LOG110 for p in e1):\n        v = \'判定不能\'\n    else:\n        v = \'当たる\' if all(p[\'ok\'] for p in e1) else \'外れる\'\n    out[\'E1\'] = dict(pairs=e1, verdict=v)\n    # E2\n    rows = {}\n    for n, m in have.items():\n        others = [k for k in have if models[k][\'fam\'] != m[\'fam\']]\n        B = float(np.mean([T0s[k] for k in others])) if others else float(\'nan\')\n        C = m[\'a\'] * float(np.mean([T0s[k] / models[k][\'a\'] for k in others])) if others else float(\'nan\')\n        T0 = T0s[n]; lo, hi = m[\'fb\'][\'T0_ci\']\n        narrow = bool(np.isfinite(lo) and hi / lo <= 1.15)\n        rows[n] = dict(T0=T0, Tcal=m[\'Tcal\'], err_E=abs(math.log(T0 / m[\'Tcal\'])), err_B=abs(math.log(T0 / B)),\n                       err_C=abs(math.log(T0 / C)), err_D=abs(math.log(T0)), ci=[lo, hi], narrow_ci=narrow,\n                       within=bool(abs(math.log(T0 / m[\'Tcal\'])) <= LOG110 or (narrow and lo <= m[\'Tcal\'] <= hi)))\n    if rows:\n        mean = {k: float(np.mean([r[\'err_\' + k] for r in rows.values()])) for k in \'EBCD\'}\n        compare = [\'B\', \'C\'] + ([] if undec_D else [\'D\'])\n        beats = all(mean[\'E\'] < mean[k] for k in compare)\n        all_within = all(r[\'within\'] for r in rows.values())\n        v = \'判定不能\' if undec_range else (\'当たる\' if (beats and all_within) else \'外れる\')\n        out[\'E2\'] = dict(rows=rows, mean_err=mean, compared_with=compare, beats_baselines=beats,\n                         all_within=all_within, n_models=len(rows), verdict=v,\n                         excluded_no_crossing=[n for n in models if n not in have])\n    else:\n        out[\'E2\'] = dict(verdict=\'交差のあるモデルがない\', excluded_no_crossing=list(models))\n    # P2\n    names = list(models)\n    fams = [models[n][\'fam\'] for n in names]\n    Fs = [np.array(models[n][\'fb\'][\'F\']) for n in names]\n    x1 = [np.log(np.array(temps)) for n in names]\n    x3 = [np.array(models[n][\'ell_tf\']) for n in names]\n    sigma = float(np.median([models[n][\'fb\'][\'eps\'] / 1.96 for n in names]))\n    e1_, o1 = spline_error(x1, Fs, fams, sigma)\n    e3_, o3 = spline_error(x3, Fs, fams, sigma)\n    v = \'判定不能\' if undec_range else (\'成り立つ\' if e3_ < 0.7 * e1_ else \'成り立たない\')\n    out[\'P2\'] = dict(err_x1=e1_, err_x3=e3_, ratio=e3_ / e1_ if e1_ > 0 else float(\'nan\'), sigma=sigma,\n                     outside_x1=o1, outside_x3=o3, verdict=v)\n    out[\'Q1_crossing\'] = {n: bool(n in have) for n in models}\n    base_T0 = [T0s[n] for n in have if models[n][\'kind\'] == \'base\']\n    out[\'Q5_base_T0_median\'] = float(np.median(base_T0)) if base_T0 else float(\'nan\')\n    out[\'tag\'] = q_tag\n    return out\n\n\ndef analyze(d):\n    rng = np.random.default_rng(SEED)\n    res = {\'models\': {}}\n    raw = {}\n    for name, fam, kind in MODELS:\n        Z = np.load(f\'{d}/{name}.npz\')\n        temps = [float(t) for t in Z[\'temps\']]\n        tf, gen = Z[\'tf\'], Z[\'gen\']\n        Tcal, n_bad = t_cal(Z[\'tcal\'], Z[\'tcal_T\'])\n        a = float(np.median(Z[\'a\'])); h = float(np.median(Z[\'h\']))\n        m = dict(fam=fam, kind=kind, Tcal=Tcal, tcal_bad_docs=n_bad, a=a, h=h, n_docs=int(tf.shape[1]),\n                 H2=int(Z[\'H2\']), tok_len=int(Z[\'tok_len\']))\n        for q, key in [(0, \'H\'), (3, \'H2\')]:\n            fd = per_doc_feedback(tf[q], gen[q])\n            m[\'fb_\' + key] = feedback_stats(fd, temps, rng)\n            m[\'ell_tf_\' + key] = np.median(tf[q].astype(np.float64).reshape(-1, len(temps), tf.shape[-1])\n                                           .transpose(1, 0, 2).reshape(len(temps), -1), 1).tolist()\n        # 記録：1 位の log-odds とエントロピーの F、f(t) の曲線、distinct-4\n        for q, key in [(1, \'top1\'), (2, \'ent\')]:\n            fd = per_doc_feedback(tf[q], gen[q])\n            m[\'F_\' + key] = np.median(fd, 0).tolist()\n        g = gen[0].astype(np.float64).mean(1).transpose(1, 0, 2) - tf[0].astype(np.float64)   # (nD, nT, L)\n        bins = [(1, 8), (8, 32), (32, 128), (128, 512)]\n        m[\'f_by_position\'] = {f\'{a_}-{b_}\': np.median(g[:, :, a_:b_].mean(-1), 0).tolist() for a_, b_ in bins}\n        m[\'distinct4_last128\'] = distinct4(Z[\'toks\'])\n        m[\'ell_gen_mean\'] = gen[0].astype(np.float64)[:, :, :, 1:].mean((1, 2, 3)).tolist()   # (nT,)\n        res[\'models\'][name] = m\n        raw[name] = m\n    res[\'temps\'] = temps\n    for key in [\'H\', \'H2\']:\n        mm = {n: dict(fam=m[\'fam\'], kind=m[\'kind\'], fb=m[\'fb_\' + key], Tcal=m[\'Tcal\'], a=m[\'a\'], ell_tf=m[\'ell_tf_\' + key])\n              for n, m in raw.items()}\n        res[\'judge_\' + key] = judge(mm, temps, key)\n    # Q6：別のモデルの文の対照\n    A_all, B_all, rows = [], [], {}\n    for reader, writer in CROSS:\n        C = np.load(f\'{d}/cross_{reader}_reads_{writer}.npz\')[\'out\']           # (2, nT, S, nD)\n        cross = np.nanmean(C[0], (1, 2)); human = np.nanmean(C[1], (1, 2))\n        self_ = np.array(raw[reader][\'ell_gen_mean\'])\n        A = self_ - cross; B = cross - human\n        rows[f\'{reader}<-{writer}\'] = dict(self_minus_cross=A.tolist(), cross_minus_human=B.tolist())\n        A_all.append(np.abs(A)); B_all.append(np.abs(B))\n    A_m, B_m = float(np.mean(A_all)), float(np.mean(B_all))\n    if abs(A_m - B_m) <= 0.2 * max(A_m, B_m):\n        v = \'同じくらい\'\n    else:\n        v = \'自分の文のほうが大きい\' if A_m > B_m else \'変な文だからのほうが大きい\'\n    res[\'Q6\'] = dict(rows=rows, mean_abs_self=A_m, mean_abs_odd=B_m, verdict=v)\n    return res\n\n\ndef _clean(o):\n    if isinstance(o, dict):\n        return {k: _clean(v) for k, v in o.items()}\n    if isinstance(o, (list, tuple)):\n        return [_clean(v) for v in o]\n    if isinstance(o, (np.floating, float)):\n        return None if not np.isfinite(o) else float(f\'{float(o):.6g}\')\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, np.bool_):\n        return bool(o)\n    return o\n\n\nif False:\n    r = analyze(sys.argv[1])\n    json.dump(_clean(r), open(sys.argv[2], \'w\'), ensure_ascii=False, separators=(\',\', \':\'))\n    print(\'wrote\', sys.argv[2])\n'
SRCA23 = '"""A2-3（ゲート v0.2、凍結 2026-10-28 14:15）の分析。scripts/a2_2_analyze.py を実行したのと同じ名前空間で実行する\n（pav・crossing_mono・feedback_stats・per_doc_feedback・t_cal・distinct4 を変えずに使う）。データを開く前に固定する。\n\n    analyze3(a2_3 のフォルダ, A2-2 の qwen2.5-1.5b.npz のパス) -> dict\n\n細部（データを開く前に決めた）：\n- 乱数：feedback_stats の種は条件ごとに np.random.default_rng(0) から作り直す（条件の順番で結果が変わらないように）。\n- 参考の判定で、交差がなく F が ±ε にとどまるときは、当てはめた F の絶対値がいちばん小さい格子の温度を点とする。\n- A100 の確かめ (b)：記事の番号を共有したブートストラップ 2,000 回（種 1）。各回で、両方の F を同じ記事の組で作り、\n  それぞれの全データの ε で T_0 を求め、log 比をとる。両方とも交差した回だけで 95% 区間を作る。\n- ε を固定した T_0（記録）：ε ＝ 0.194（A2-2 の 7 モデルの ε の中央値）。\n- コードの探索：記事ごとの NLL がいちばん低い 4 分の 1（10 本）を除いた 30 本で T_0（同じ規則）。\n"""\nimport math\nimport numpy as np\n\nBAND = (0.78, 0.93)\nCI_RATIO_MAX, CROSS_FRAC_MIN = 1.20, 0.9\nEPS_FIXED = 0.194\nA22_BBC_T0 = {\'qwen2.5-1.5b\': 0.910086, \'smollm2-1.7b\': 0.800452, \'pythia-1b\': 0.852045}   # results/A2_2/a2_2_summary.json\nJUDGED = [\'qwen2.5-7b@bbc\', \'olmo2-7b@bbc\', \'qwen2.5-1.5b@code\', \'smollm2-1.7b@code\', \'pythia-1b@code\']\n\n\ndef main_verdict(fb, temps):\n    T0, (lo, hi), frac, eps = fb[\'T0\'], fb[\'T0_ci\'], fb[\'boot_cross_frac\'], fb[\'eps\']\n    g = np.asarray(fb[\'F_pav\'])\n    if np.isfinite(T0):\n        inside = BAND[0] <= T0 <= BAND[1]\n        ci_ok = np.isfinite(lo) and np.isfinite(hi)\n        if inside and ci_ok and hi / lo <= CI_RATIO_MAX and frac >= CROSS_FRAC_MIN:\n            return \'帯に入る\'\n        if not inside and ci_ok and (hi < BAND[0] or lo > BAND[1]):\n            return \'帯の外（上）\' if T0 > BAND[1] else \'帯の外（下）\'\n        return \'はっきりしない\'\n    if g[0] >= eps:\n        return \'帯の外（下）\'\n    if g[-1] <= -eps:\n        return \'帯の外（上）\'\n    return \'はっきりしない\'\n\n\ndef ref_point(fb, temps):\n    T0, eps = fb[\'T0\'], fb[\'eps\']\n    g = np.asarray(fb[\'F_pav\'])\n    if np.isfinite(T0):\n        return T0, \'T0\'\n    if g[0] >= eps:\n        return float(\'-inf\'), \'below_grid\'\n    if g[-1] <= -eps:\n        return float(\'inf\'), \'above_grid\'\n    return float(temps[int(np.argmin(np.abs(g)))]), \'closest_to_zero\'\n\n\ndef ref_verdict(p):\n    if BAND[0] <= p <= BAND[1]:\n        return \'帯に入る\'\n    return \'帯の外（上）\' if p > BAND[1] else \'帯の外（下）\'\n\n\ndef a100_check(Z_new, Z_old, temps):\n    assert np.array_equal(Z_new[\'use\'], Z_old[\'use\']) and Z_new[\'tf\'].shape == Z_old[\'tf\'].shape, \'A100 の確かめ：記事の組が A2-2 と違う\'\n    out = {\'revision_same\': str(Z_new[\'revision\']) == str(Z_old[\'revision\'])}\n    out[\'tf_max_abs_diff\'] = float(np.max(np.abs(Z_new[\'tf\'][0].astype(np.float64) - Z_old[\'tf\'][0].astype(np.float64))))\n    Tn, _ = t_cal(Z_new[\'tcal\'], Z_new[\'tcal_T\']); To, _ = t_cal(Z_old[\'tcal\'], Z_old[\'tcal_T\'])\n    out[\'Tcal_new\'], out[\'Tcal_old\'], out[\'Tcal_abs_diff\'] = Tn, To, abs(Tn - To)\n    fn = per_doc_feedback(Z_new[\'tf\'][0], Z_new[\'gen\'][0]); fo = per_doc_feedback(Z_old[\'tf\'][0], Z_old[\'gen\'][0])\n    sn = feedback_stats(fn, temps, np.random.default_rng(0)); so = feedback_stats(fo, temps, np.random.default_rng(0))\n    out[\'T0_new\'], out[\'T0_old\'] = sn[\'T0\'], so[\'T0\']\n    rng = np.random.default_rng(1)\n    nD = fn.shape[0]\n    lr = []\n    for _ in range(NBOOT):\n        idx = rng.integers(0, nD, nD)\n        a = crossing_mono(np.median(fn[idx], 0), temps, sn[\'eps\'])\n        b = crossing_mono(np.median(fo[idx], 0), temps, so[\'eps\'])\n        if np.isfinite(a) and np.isfinite(b):\n            lr.append(math.log(a / b))\n    lr = np.array(lr)\n    point = math.log(sn[\'T0\'] / so[\'T0\']) if np.isfinite(sn[\'T0\']) and np.isfinite(so[\'T0\']) else float(\'nan\')\n    ci = [float(np.percentile(lr, 2.5)), float(np.percentile(lr, 97.5))] if len(lr) >= 20 else [float(\'nan\')] * 2\n    out.update(log_ratio=point, log_ratio_ci=ci, boot_both_cross=len(lr) / NBOOT)\n    ok_a = out[\'tf_max_abs_diff\'] <= 1e-3 and out[\'Tcal_abs_diff\'] <= 1e-3\n    ok_b = (np.isfinite(ci[0]) and ci[0] <= 0 <= ci[1]) or (np.isfinite(point) and abs(point) <= 0.03)\n    out.update(ok_a=bool(ok_a), ok_b=bool(ok_b), ok=bool(ok_a and ok_b))\n    return out\n\n\ndef analyze3(d, a22_qwen15_npz):\n    res = {\'conditions\': {}}\n    temps = None\n    for name in [\'qwen2.5-1.5b@bbc-a100\'] + JUDGED:\n        Z = np.load(f\'{d}/{name}.npz\')\n        temps = [float(t) for t in Z[\'temps\']]\n        Tcal, nbad = t_cal(Z[\'tcal\'], Z[\'tcal_T\'])\n        c = {\'Tcal\': Tcal, \'tcal_bad_docs\': nbad, \'n_docs\': int(Z[\'tf\'].shape[1]), \'chunk\': int(Z[\'chunk\'])}\n        for q, key in [(0, \'H\'), (3, \'H2\')]:\n            fd = per_doc_feedback(Z[\'tf\'][q], Z[\'gen\'][q])\n            fb = feedback_stats(fd, temps, np.random.default_rng(0))\n            c[\'fb_\' + key] = fb\n            if key == \'H\':\n                c[\'T0_eps_fixed\'] = crossing_mono(fb[\'F\'], temps, EPS_FIXED)\n                if name.endswith(\'@code\'):\n                    keep = np.argsort(Z[\'nll\'])[len(Z[\'nll\']) // 4:]\n                    c[\'T0_without_lowest_nll_quarter\'] = feedback_stats(fd[np.sort(keep)], temps, np.random.default_rng(0))[\'T0\']\n        for q, key in [(1, \'top1\'), (2, \'ent\')]:\n            c[\'F_\' + key] = np.median(per_doc_feedback(Z[\'tf\'][q], Z[\'gen\'][q]), 0).tolist()\n        c[\'distinct4_last128\'] = distinct4(Z[\'toks\'])\n        c[\'nll\'] = Z[\'nll\'].tolist()\n        c[\'ext\'] = [str(x) for x in Z[\'ext\']]\n        c[\'main\'] = main_verdict(c[\'fb_H\'], temps)\n        p, how = ref_point(c[\'fb_H\'], temps)\n        c[\'ref_point\'], c[\'ref_point_how\'], c[\'ref\'] = p, how, ref_verdict(p)\n        res[\'conditions\'][name] = c\n    res[\'temps\'] = temps\n    res[\'a100_check\'] = a100_check(np.load(f\'{d}/qwen2.5-1.5b@bbc-a100.npz\'), np.load(a22_qwen15_npz), temps)\n    mains = [res[\'conditions\'][n][\'main\'] for n in JUDGED]\n    if not res[\'a100_check\'][\'ok\']:\n        overall = \'はっきりしない（A100 の確かめが通らなかった）\'\n    elif all(m == \'帯に入る\' for m in mains):\n        overall = \'一般的\'\n    elif any(m.startswith(\'帯の外\') for m in mains):\n        overall = \'一般的ではない\'\n    else:\n        overall = \'はっきりしない\'\n    refs = [res[\'conditions\'][n][\'ref\'] for n in JUDGED]\n    res[\'overall_main\'] = overall\n    res[\'overall_ref\'] = \'一般的\' if all(r == \'帯に入る\' for r in refs) else \'一般的ではない\'\n    # 軸ごとの読み方（ゲート §2）\n    size = [res[\'conditions\'][n][\'main\'] for n in JUDGED[:2]]\n    outs = [m for m in size if m.startswith(\'帯の外\')]\n    res[\'axis_size\'] = (\'大きさによる（\' + outs[0] + \'）\' if len(outs) == 2 and outs[0] == outs[1]\n                        else \'2 つとも外れる（向きが逆）\' if len(outs) == 2\n                        else \'その家族では外れる\' if outs else \'外れなし\')\n    code = [res[\'conditions\'][n][\'main\'] for n in JUDGED[2:]]\n    outc = [m for m in code if m.startswith(\'帯の外\')]\n    res[\'axis_text\'] = (\'文章の種類による（\' + outc[0] + \'）\' if len(outc) == 3 and len(set(outc)) == 1\n                        else \'モデルと文章の組み合わせによる\' if outc else \'外れなし\')\n    # 採点用\n    res[\'Q3_code_in_band\'] = sum(m == \'帯に入る\' for m in code)\n    moves = {}\n    for n in JUDGED[2:]:\n        base = A22_BBC_T0[n.split(\'@\')[0]]\n        p = res[\'conditions\'][n][\'ref_point\']\n        r = p / base if np.isfinite(p) else (float(\'inf\') if p > 0 else 0.0)\n        moves[n] = \'上がる\' if r > 1.03 else \'下がる\' if r < 0.97 else \'ほぼ同じ\'\n    vals = list(moves.values())\n    res[\'Q4_moves\'] = moves\n    res[\'Q4\'] = max(set(vals), key=vals.count) if max(vals.count(v) for v in set(vals)) >= 2 else \'ばらばら\'\n    return res\n'
assert hashlib.sha256(SRCA22.encode()).hexdigest() == 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7'
assert hashlib.sha256(SRCA23.encode()).hexdigest() == '6003f2fc37ea03aebc53a5e4eb86728de2cc4f314671ba8c49a8843d88cad761'
exec(SRCA22)
exec(SRCA23)
print('コードを置きました')


In [ ]:
# セル3：分析して保存（結果は表示しない）
import json
r = analyze3('/content/drive/MyDrive/logit-thermo/A2_3', '/content/drive/MyDrive/logit-thermo/A2_2/qwen2.5-1.5b.npz')
r['a2_3_analyze_sha256'] = '6003f2fc37ea03aebc53a5e4eb86728de2cc4f314671ba8c49a8843d88cad761'
r['a2_2_analyze_sha256'] = '35142703f6b88d7a001bfe5ff3608ea9c6cb2e60c457fc02beb235fba946f20e'
json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_3' + '/a2_3_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
print('保存しました')


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：判定と要約を a2_3_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
